# SHBP State of Georgia — Member Progress Report

**Sections:**
1. Setup & DB connection
2. Pull all members from DB + run tasks, conditions, derived columns
3. Load & deduplicate PA expiry files → match to members
4. Build full report Excel
5. Quick sanity checks

**Folder structure expected (create these next to this notebook):**
```
pa_files/        ← drop monthly PA expiry Excel files here
claims_files/    ← drop AOM/claims Excel files here (optional)
output/          ← report files will be saved here
```

## 1. Setup

In [1]:
import os
os.environ['DB_ENV'] = 'prod'
import  sys, re, warnings, importlib.util
import pandas as pd
import numpy as np
from datetime import date, datetime
from pathlib import Path
import mysql.connector

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

PA_FILES_DIR     = Path('pa_files')
CLAIMS_FILES_DIR = Path('claims_files')
OUTPUT_DIR       = Path('output')
for d in [PA_FILES_DIR, CLAIMS_FILES_DIR, OUTPUT_DIR]:
    d.mkdir(exist_ok=True)

TODAY         = pd.Timestamp(date.today())
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'✅ Setup complete  |  Run date: {TODAY.date()}')

✅ Setup complete  |  Run date: 2026-03-04


In [2]:
try:
    from config import get_db_config
except ImportError:
    sys.exit('❌ config.py not found — place it in the same directory as this notebook')

def connect_to_db():
    cfg = get_db_config()
    cfg['connect_timeout'] = 300
    return mysql.connector.connect(**cfg)

conn = connect_to_db()
print('✅ Connected to DB')

✅ Connected to DB


## 2. Pull Members from DB

In [3]:
# Load the main script as a module
spec = importlib.util.spec_from_file_location('gcf', 'georgia_continuation_full.py')
gcf  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gcf)

print('📥 Running main DB query...')
df = gcf.run_main_query(conn)
print(f'\n  Rows returned:      {len(df):,}')
print(f'  Unique readable_id: {df["readable_id"].nunique():,}')

📥 Running main DB query...
  📥 Fetching Georgia continuation members (full detail)...
    ...9,316 rows
    ✅ 9,316 rows in 30.4s

  Rows returned:      9,316
  Unique readable_id: 9,316


In [4]:
member_ids = df['member_id'].tolist()

print('📥 Fetching tasks...')
task_df = gcf.run_task_analysis(conn, member_ids)
df = gcf.merge_tasks(df, task_df)
df = gcf.add_task_summary_cols(df)

print('\n📥 Fetching medical conditions...')
cond_df      = gcf.get_medical_conditions(conn, member_ids)
cond_summary = gcf.summarize_conditions(cond_df)
df = pd.merge(df, cond_summary, on='member_id', how='left')

df = gcf.add_derived_columns(df)
df['member_category'] = df.apply(gcf.assign_member_category, axis=1)

print(f'\n✅ Full dataset ready: {len(df):,} members')
print('\nMember category breakdown:')
print(df['member_category'].value_counts().to_string())

📥 Fetching tasks...

  📥 Fetching tasks for 9,316 members...
    ✅ 9,315 members with task records in 4.2s

📥 Fetching medical conditions...
    ✅ 36,011 condition records

✅ Full dataset ready: 9,316 members

Member category breakdown:
member_category
Not GLP-1 Continuation Member             3012
Has WM Rx from 9amHealth                  2727
No WM Rx — Questionnaire Complete Only    2597
No WM Rx — No Tasks Complete               601
No WM Rx — Both Tasks Complete             366
No WM Rx — Lab Complete Only                13


## 3. Load & Deduplicate PA Expiry Files

In [7]:
# ── PA file loader — handles all 4 file formats automatically ───────────────
# Feb/Mar: named headers, has DOB — match on last+first+DOB
# Apr:     no header row (positional) — has DOB — match on last+first+DOB
# May:     simple format, NO DOB — match on last+first+zip

MONTH_MAP = {
    'jan': 'Jan 2026', 'feb': 'Feb 2026', 'mar': 'Mar 2026',
    'apr': 'Apr 2026', 'may': 'May 2026', 'jun': 'Jun 2026',
    'jul': 'Jul 2026', 'aug': 'Aug 2026', 'sep': 'Sep 2026',
    'oct': 'Oct 2026', 'nov': 'Nov 2026', 'dec': 'Dec 2026',
}

PA_COLUMN_MAP = {
    'Member First Name':                       'pa_first_name',
    'Member Last Name':                        'pa_last_name',
    'Member Birth Date - Current':             'pa_dob',
    'Member Zip Code5':                        'pa_zip',
    'Member Address Line1':                    'pa_address',
    'Member City':                             'pa_city',
    'Member State':                            'pa_state',
    'Member Account ID':                       'pa_member_account_id',
    'Member Account External ID - Current':    'pa_member_external_id',
    'Product Name':                            'pa_drug_name',
    'Prior Authorization NDC GPI Description': 'pa_drug_description',
    'Prior Authorization Expiration Date':     'pa_expiry_date',
    'Prior Authorization Effective Date':      'pa_effective_date',
    'Prior Authorization Number':              'pa_auth_number',
    # May file
    'First Name':   'pa_first_name',
    'Last Name':    'pa_last_name',
    'Zip':          'pa_zip',
    'Address 1':    'pa_address',
    'City':         'pa_city',
    'State':        'pa_state',
    'Expiry Month': 'pa_expiry_month_col',
    'Unique ID':    'pa_unique_id',
}

# April: no header row — column positions
APR_POSITIONAL_COLS = {
    6:  'pa_member_external_id',
    7:  'pa_member_account_id',
    8:  'pa_first_name',
    9:  'pa_last_name',
    10: 'pa_age',
    11: 'pa_dob',
    12: 'pa_address',
    14: 'pa_city',
    15: 'pa_state',
    16: 'pa_zip',
    20: 'pa_drug_description',
    21: 'pa_drug_name',
    22: 'pa_effective_date',
    23: 'pa_expiry_date',
}

def detect_month(fname):
    return next((v for k, v in MONTH_MAP.items() if k in fname.lower()), 'Unknown')

def load_pa_file(path):
    fname = Path(path).stem
    expiry_month = detect_month(fname)
    is_april = 'april' in fname.lower() or ('apr' in fname.lower() and 'mar' not in fname.lower())
    is_may   = 'may' in fname.lower()

    if is_april:
        raw = pd.read_excel(path, header=None, dtype=str)
        df  = raw.rename(columns=APR_POSITIONAL_COLS)
        df  = df[[v for v in APR_POSITIONAL_COLS.values() if v in df.columns]]
    else:
        df = pd.read_excel(path, dtype=str)
        df.columns = df.columns.str.strip()
        df = df.rename(columns={k: v for k, v in PA_COLUMN_MAP.items() if k in df.columns})

    df['pa_source_file']  = Path(path).name
    df['pa_expiry_month'] = expiry_month
    df['pa_has_dob']      = (0 if is_may else 1)  # flag: can we use DOB for matching?

    if 'pa_expiry_month_col' in df.columns:
        df['pa_expiry_month'] = df['pa_expiry_month_col'].apply(
            lambda x: detect_month(str(x)) if pd.notna(x) else expiry_month
        )
        df = df.drop(columns=['pa_expiry_month_col'])

    if 'pa_dob' in df.columns:
        df['pa_dob'] = pd.to_datetime(df['pa_dob'], errors='coerce').dt.strftime('%Y-%m-%d')

    print(f'  {Path(path).name}: {len(df):,} rows | month={expiry_month} | has_dob={not is_may}')
    return df

def load_pa_files(pa_dir):
    files = sorted(list(pa_dir.glob('*.xlsx')) + list(pa_dir.glob('*.xls')))
    if not files:
        print(f'⚠️  No Excel files found in {pa_dir.resolve()}')
        return pd.DataFrame()
    dfs = [load_pa_file(f) for f in files]
    df_pa = pd.concat(dfs, ignore_index=True)
    print(f'\n  Total raw PA rows: {len(df_pa):,}')
    return df_pa

df_pa_raw = load_pa_files(PA_FILES_DIR)
print('\nMonth distribution:')
print(df_pa_raw['pa_expiry_month'].value_counts().sort_index().to_string())


  April Expiring PAs.xlsx: 3,860 rows | month=Apr 2026 | has_dob=True
  FEb Expiring PA in Member 90 Day History.xlsx: 4,129 rows | month=Feb 2026 | has_dob=True
  Mar Expiring PA in Member 90 Day History.xlsx: 4,327 rows | month=Mar 2026 | has_dob=True
  May Expiring PAs.xlsx: 4,239 rows | month=May 2026 | has_dob=False

  Total raw PA rows: 16,555

Month distribution:
pa_expiry_month
Apr 2026    3860
Feb 2026    4129
Mar 2026    4327
May 2026    4239


In [8]:
import re

def normalize_name(s):
    if pd.isna(s): return ''
    return re.sub(r'[^A-Z0-9]', '', str(s).upper().strip())

def normalize_dob(s):
    try:
        return pd.to_datetime(s, errors='coerce').strftime('%Y-%m-%d')
    except:
        return ''

def deduplicate_pa_members(df_pa):
    """
    Members can appear across monthly files.
    Key strategy:
      - Feb/Mar/Apr (has DOB): last + first + dob
      - May (no DOB):          last + first + zip
    Keep latest expiry month per member. Track all files they appeared in.
    """
    if df_pa.empty:
        return df_pa

    df = df_pa.copy()

    fn  = 'pa_first_name' if 'pa_first_name' in df.columns else None
    ln  = 'pa_last_name'  if 'pa_last_name'  in df.columns else None
    dob = 'pa_dob'        if 'pa_dob'        in df.columns else None
    zipc= 'pa_zip'        if 'pa_zip'        in df.columns else None

    # Build dedup key: use DOB when available, zip as fallback for May rows
    has_dob = df.get('pa_has_dob', pd.Series(1, index=df.index)).fillna(1).astype(int)
    dob_vals = df[dob].apply(normalize_dob) if dob else pd.Series('', index=df.index)
    zip_vals = df[zipc].apply(normalize_name) if zipc else pd.Series('', index=df.index)
    fn_vals  = df[fn].apply(normalize_name)   if fn   else pd.Series('', index=df.index)
    ln_vals  = df[ln].apply(normalize_name)   if ln   else pd.Series('', index=df.index)

    df['_dedup_key'] = np.where(
        has_dob == 1,
        ln_vals + '|' + fn_vals + '|' + dob_vals,
        ln_vals + '|' + fn_vals + '|ZIP|' + zip_vals
    )

    # Track all months and files per member
    all_months = df.groupby('_dedup_key')['pa_expiry_month'].apply(
        lambda x: ', '.join(sorted(x.dropna().unique()))
    ).reset_index().rename(columns={'pa_expiry_month': 'pa_all_expiry_months'})

    all_files = df.groupby('_dedup_key')['pa_source_file'].apply(
        lambda x: ', '.join(sorted(x.dropna().unique()))
    ).reset_index().rename(columns={'pa_source_file': 'pa_all_source_files'})

    # Keep the row with the latest expiry month
    month_order = {v: i for i, v in enumerate(MONTH_MAP.values())}
    df['_month_rank'] = df['pa_expiry_month'].map(month_order).fillna(99)
    df_dedup = df.sort_values('_month_rank').groupby('_dedup_key', as_index=False).last()
    df_dedup = df_dedup.merge(all_months, on='_dedup_key', how='left')
    df_dedup = df_dedup.merge(all_files,  on='_dedup_key', how='left')
    df_dedup['appeared_in_multiple_pa_files'] = (
        df_dedup['pa_all_source_files'].str.contains(',').astype(int)
    )
    df_dedup = df_dedup.drop(columns=['_dedup_key', '_month_rank'])

    print(f'  Raw PA rows:                {len(df_pa):,}')
    print(f'  Unique members after dedup: {len(df_dedup):,}')
    print(f'  Appeared in multiple files: {df_dedup["appeared_in_multiple_pa_files"].sum():,}')
    return df_dedup

if not df_pa_raw.empty:
    df_pa = deduplicate_pa_members(df_pa_raw)
    print('\nLatest expiry month (after dedup):')
    print(df_pa['pa_expiry_month'].value_counts().sort_index().to_string())
else:
    df_pa = pd.DataFrame()
    print('⚠️  No PA files loaded')


  Raw PA rows:                16,555
  Unique members after dedup: 16,450
  Appeared in multiple files: 7

Latest expiry month (after dedup):
pa_expiry_month
Apr 2026    3833
Feb 2026    4086
Mar 2026    4292
May 2026    4239


In [9]:
def match_pa_to_members(df_members, df_pa):
    """
    Match PA file → DB members.
    Primary:  last + first + dob
    Fallback: last + dob + zip

    Returns df_pa with match columns added.
    Also returns:
      - df_pa_matched:   PA members found in 9am DB
      - df_pa_unmatched: PA members NOT found in 9am DB
      - df_members_not_in_pa: 9am members NOT in any PA file
    """
    if df_pa.empty:
        return df_pa, pd.DataFrame(), pd.DataFrame(), df_members

    MATCH_COLS = [
        'member_id','readable_id','first_name','last_name','member_category',
        'has_wm_rx_since_2026','latest_wm_rx_drug_name','latest_wm_rx_2026_date',
        'first_glp1_rx_date','first_glp1_rx_drug_name',
        'reported_medication_name','prescription_id','prescribed_medication_name',
        'subscription_start_date','days_enrolled',
        'lab_done','questionnaire_done','both_tasks_done','tasks_completed_count',
    ]
    MATCH_COLS = [c for c in MATCH_COLS if c in df_members.columns]

    m = df_members[MATCH_COLS + ['date_of_birth','shipping_zip']].copy()
    m['_key_full'] = (
        m['last_name'].apply(normalize_name) + '|' +
        m['first_name'].apply(normalize_name) + '|' +
        m['date_of_birth'].apply(normalize_dob)
    )
    m['_key_fallback'] = (
        m['last_name'].apply(normalize_name) + '|' +
        m['date_of_birth'].apply(normalize_dob) + '|' +
        m['shipping_zip'].apply(normalize_name)
    )

    p = df_pa.copy()
    fn  = 'pa_first_name' if 'pa_first_name' in p.columns else None
    ln  = 'pa_last_name'  if 'pa_last_name'  in p.columns else None
    dob = 'pa_dob'        if 'pa_dob'        in p.columns else None
    zipc= 'pa_zip'        if 'pa_zip'        in p.columns else None

    p['_key_full'] = (
        (p[ln].apply(normalize_name)  if ln  else '') + '|' +
        (p[fn].apply(normalize_name)  if fn  else '') + '|' +
        (p[dob].apply(normalize_dob)  if dob else '')
    )
    p['_key_fallback'] = (
        (p[ln].apply(normalize_name)  if ln   else '') + '|' +
        (p[dob].apply(normalize_dob)  if dob  else '') + '|' +
        (p[zipc].apply(normalize_name) if zipc else '')
    )

    # Primary match
    lkp_full = m.drop_duplicates('_key_full').set_index('_key_full')[MATCH_COLS]
    p = p.join(lkp_full.add_prefix('match_'), on='_key_full')

    # Fallback for unmatched
    unmatched_mask = p['match_member_id'].isna()
    if unmatched_mask.any():
        lkp_fb = m.drop_duplicates('_key_fallback').set_index('_key_fallback')[MATCH_COLS]
        for col in MATCH_COLS:
            fb_vals = p.loc[unmatched_mask, '_key_fallback'].map(lkp_fb[col])
            p.loc[unmatched_mask, f'match_{col}'] = fb_vals

    p['matched_to_9am_member'] = p['match_member_id'].notna().astype(int)
    p['match_method'] = np.where(
        p['matched_to_9am_member'] == 0, 'no_match',
        np.where(p['_key_full'].isin(lkp_full.index), 'full_name_dob', 'fallback_name_dob_zip')
    )

    # Same med vs switched: PA drug vs reported survey drug
    if 'pa_drug_name' in p.columns:
        pa_drug_norm = p['pa_drug_name'].fillna('').str.lower().str.strip()
        rx_drug_norm = p['match_reported_medication_name'].fillna('').str.lower().str.strip()
        p['pa_vs_reported_same_med'] = np.where(
            (pa_drug_norm == '') | (rx_drug_norm == ''), 'Unknown',
            np.where(pa_drug_norm == rx_drug_norm, 'Same Med', 'Different Med')
        )
        # vs what they were actually prescribed
        presc_norm = p['match_prescribed_medication_name'].fillna('').str.lower().str.strip()
        p['pa_vs_prescribed_same_med'] = np.where(
            (pa_drug_norm == '') | (presc_norm == ''), 'Unknown',
            np.where(pa_drug_norm == presc_norm, 'Same Med', 'Switched')
        )

    p = p.drop(columns=['_key_full','_key_fallback'])

    df_pa_matched   = p[p['matched_to_9am_member'] == 1].copy()
    df_pa_unmatched = p[p['matched_to_9am_member'] == 0].copy()

    # 9am members NOT in PA file
    matched_9am_ids = set(df_pa_matched['match_member_id'].dropna())
    df_members_not_in_pa = df_members[
        ~df_members['member_id'].isin(matched_9am_ids)
    ].copy()

    return p, df_pa_matched, df_pa_unmatched, df_members_not_in_pa

if not df_pa.empty:
    df_pa_full, df_pa_matched, df_pa_unmatched, df_members_not_in_pa = \
        match_pa_to_members(df, df_pa)

    print(f'\n✅ PA matching complete')
    print(f'  PA members matched to 9am DB:   {len(df_pa_matched):,}')
    print(f'  PA members NOT in 9am DB:       {len(df_pa_unmatched):,}')
    print(f'  9am members NOT in any PA file: {len(df_members_not_in_pa):,}')
    if 'match_has_wm_rx_since_2026' in df_pa_matched.columns:
        has_rx = df_pa_matched['match_has_wm_rx_since_2026'].fillna(0).astype(int)
        print(f'\n  Of matched PA members:')
        print(f'    Has WM Rx from 9am:  {has_rx.sum():,}')
        print(f'    No WM Rx yet:        {(~has_rx.astype(bool)).sum():,}')
else:
    df_pa_full = df_pa_matched = df_pa_unmatched = df_members_not_in_pa = pd.DataFrame()
    print('⚠️  No PA files — skipping match')


✅ PA matching complete
  PA members matched to 9am DB:   3,673
  PA members NOT in 9am DB:       12,777
  9am members NOT in any PA file: 5,663

  Of matched PA members:
    Has WM Rx from 9am:  1,541
    No WM Rx yet:        2,132


In [10]:
# ── Per-month PA summary table ────────────────────────────────────────────────
def build_pa_month_summary(df_pa_full):
    if df_pa_full.empty or 'pa_expiry_month' not in df_pa_full.columns:
        return pd.DataFrame()
    rows = []
    for month in sorted(df_pa_full['pa_expiry_month'].dropna().unique()):
        grp     = df_pa_full[df_pa_full['pa_expiry_month'] == month]
        matched = grp[grp['matched_to_9am_member'] == 1]
        has_rx  = matched['match_has_wm_rx_since_2026'].fillna(0).astype(int)
        rows.append({
            'PA Expiry Month':                  month,
            'Total in PA File':                 len(grp),
            'Matched to 9am Member':            len(matched),
            'Not Matched (not enrolled)':       len(grp) - len(matched),
            'Matched + Has WM Rx':              int(has_rx.sum()),
            'Matched + No WM Rx Yet':           int((~has_rx.astype(bool)).sum()),
            'Pct Matched Prescribed':           f"{has_rx.sum()/len(matched)*100:.1f}%" if len(matched) else 'N/A',
        })
    return pd.DataFrame(rows)

if not df_pa_full.empty:
    pa_month_summary = build_pa_month_summary(df_pa_full)
    display(pa_month_summary)

,PA Expiry Month,Total in PA File,Matched to 9am Member,Not Matched (not enrolled),Matched + Has WM Rx,Matched + No WM Rx Yet,Pct Matched Prescribed
0,Apr 2026,3833,644,3189,110,534,17.1%
1,Feb 2026,4086,1875,2211,986,889,52.6%
2,Mar 2026,4292,1154,3138,445,709,38.6%
3,May 2026,4239,0,4239,0,0,N/A


## 4. Build & Export Full Report

In [11]:
output_path = OUTPUT_DIR / f'georgia_shbp_report_{RUN_TIMESTAMP}.xlsx'
print(f'📤 Building report → {output_path}\n')

summary_df = gcf.build_summary(df)

# Drug switching tables
switching_result = gcf.build_drug_switching_table(df)

# Dropped off definition: no Rx + enrolled 30+ days + zero task activity
dropped_off = df[
    (df['past_glp1_use'] == 1) &
    (df['has_wm_rx_since_2026'] == 0) &
    (df['days_enrolled'] >= 30) &
    (df['tasks_completed_count'] == 0)
].copy()

# Both tasks done, no Rx yet (continuation members)
pending_both_tasks = df[
    (df['past_glp1_use'] == 1) &
    (df['both_tasks_done'] == 1) &
    (df['has_wm_rx_since_2026'] == 0)
].copy()

drop_cols = ['most_recent_dose_date_raw']
df_out = df.drop(columns=[c for c in drop_cols if c in df.columns])

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # ── Summary ───────────────────────────────────────────────────────────────
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    print('  ✅ Summary')

    # ── All members ───────────────────────────────────────────────────────────
    df_out.to_excel(writer, sheet_name='All Members', index=False)
    print(f'  ✅ All Members: {len(df_out):,}')

    # ── Per category sheets ───────────────────────────────────────────────────
    for cat in df_out['member_category'].value_counts().index:
        df_out[df_out['member_category'] == cat].to_excel(
            writer, sheet_name=cat[:31], index=False)
        print(f'  ✅ {cat}: {len(df_out[df_out["member_category"]==cat]):,}')

    # ── Dropped off ───────────────────────────────────────────────────────────
    if len(dropped_off):
        dropped_off.to_excel(writer, sheet_name='Dropped Off', index=False)
        print(f'  ✅ Dropped Off: {len(dropped_off):,}')

    # ── Both tasks done, pending Rx ───────────────────────────────────────────
    if len(pending_both_tasks):
        pending_both_tasks.to_excel(writer, sheet_name='Pending — Both Tasks Done', index=False)
        print(f'  ✅ Pending — Both Tasks Done: {len(pending_both_tasks):,}')

    # ── Drug switching ────────────────────────────────────────────────────────
    if switching_result:
        pivot, switch_summary, switch_detail = switching_result
        switch_summary.to_excel(writer, sheet_name='Drug Switching — Summary', index=False)
        pivot.to_excel(writer, sheet_name='Drug Switching — Crosstab', index=False)
        switch_detail.to_excel(writer, sheet_name='Drug Switching — Detail', index=False)
        print('  ✅ Drug Switching sheets (Summary, Crosstab, Detail)')

    # ── PA file sheets ────────────────────────────────────────────────────────
    if not df_pa_full.empty:
        # Section 5: PA Expiry Reconciliation — month summary
        pa_month_summary.to_excel(writer, sheet_name='PA — Month Summary', index=False)
        print(f'  ✅ PA — Month Summary')

        # All PA members with match status
        df_pa_full.to_excel(writer, sheet_name='PA — All (with match)', index=False)
        print(f'  ✅ PA — All (with match): {len(df_pa_full):,}')

        # Matched: in PA file AND enrolled at 9am
        df_pa_matched.to_excel(writer, sheet_name='PA — Matched (enrolled)', index=False)
        print(f'  ✅ PA — Matched (enrolled): {len(df_pa_matched):,}')

        # Unmatched: in PA file, NOT in 9am DB
        df_pa_unmatched.to_excel(writer, sheet_name='PA — Not in 9am DB', index=False)
        print(f'  ✅ PA — Not in 9am DB: {len(df_pa_unmatched):,}')

        # 9am members NOT in any PA file
        if len(df_members_not_in_pa):
            df_members_not_in_pa.to_excel(writer, sheet_name='Members — Not in PA Files', index=False)
            print(f'  ✅ Members — Not in PA Files: {len(df_members_not_in_pa):,}')

        # Section 6: Appendix — full member list with PA expiry month
        # Join PA expiry month back onto member list for the appendix
        pa_month_lookup = df_pa_full[df_pa_full['matched_to_9am_member']==1][
            ['match_member_id','pa_expiry_month','pa_all_expiry_months','appeared_in_multiple_pa_files']
        ].rename(columns={'match_member_id':'member_id'})
        appendix = df_out.merge(pa_month_lookup, on='member_id', how='left')
        appendix.to_excel(writer, sheet_name='Appendix — Full Member List', index=False)
        print(f'  ✅ Appendix — Full Member List: {len(appendix):,}')
    else:
        # No PA files — still write appendix without PA columns
        df_out.to_excel(writer, sheet_name='Appendix — Full Member List', index=False)
        print(f'  ✅ Appendix — Full Member List: {len(df_out):,} (no PA file data)')

print(f'\n✅ Report saved → {output_path}')

📤 Building report → output/georgia_shbp_report_20260304_115230.xlsx

  ✅ Summary
  ✅ All Members: 9,316
  ✅ Not GLP-1 Continuation Member: 3,012
  ✅ Has WM Rx from 9amHealth: 2,727
  ✅ No WM Rx — Questionnaire Complete Only: 2,597
  ✅ No WM Rx — No Tasks Complete: 601
  ✅ No WM Rx — Both Tasks Complete: 366
  ✅ No WM Rx — Lab Complete Only: 13
  ✅ Dropped Off: 47
  ✅ Pending — Both Tasks Done: 366
  ✅ Drug Switching sheets (Summary, Crosstab, Detail)
  ✅ PA — Month Summary
  ✅ PA — All (with match): 16,450
  ✅ PA — Matched (enrolled): 3,673
  ✅ PA — Not in 9am DB: 12,777
  ✅ Members — Not in PA Files: 5,663
  ✅ Appendix — Full Member List: 9,316

✅ Report saved → output/georgia_shbp_report_20260304_115230.xlsx


## 5. Sanity Checks

In [12]:
# ── Duplicate check ───────────────────────────────────────────────────────────
dupes = df[df.duplicated('readable_id', keep=False)]
print(f'Duplicate readable_ids: {dupes["readable_id"].nunique()} ({len(dupes)} rows)')
if len(dupes):
    display(dupes[['readable_id','first_name','last_name','member_category']].head(10))

Duplicate readable_ids: 0 (0 rows)


In [13]:
# ── Overall funnel ────────────────────────────────────────────────────────────
cont = df[df['past_glp1_use'] == 1]
print(f'Total members:              {len(df):,}')
print(f'Continuation members:       {len(cont):,}')
print(f'  Has WM Rx:                {cont["has_wm_rx_since_2026"].sum():,}')
print(f'  Both tasks, no Rx:        {((cont["both_tasks_done"]==1)&(cont["has_wm_rx_since_2026"]==0)).sum():,}')
print(f'  Dropped off:              {((cont["has_wm_rx_since_2026"]==0)&(cont["days_enrolled"]>=30)&(cont["tasks_completed_count"]==0)).sum():,}')
if len(cont):
    print(f'  Prescribed rate:          {cont["has_wm_rx_since_2026"].mean()*100:.1f}%')

Total members:              9,316
Continuation members:       6,304
  Has WM Rx:                2,727
  Both tasks, no Rx:        366
  Dropped off:              47
  Prescribed rate:          43.3%


In [14]:
# ── BMI eligibility for pending members ──────────────────────────────────────
pending = df[
    (df['past_glp1_use'] == 1) &
    (df['both_tasks_done'] == 1) &
    (df['has_wm_rx_since_2026'] == 0)
]
print(f'Both tasks done, no WM Rx: {len(pending):,}')
print(f'  BMI > 30:     {(pending["calculated_bmi"] > 30).sum():,}')
print(f'  BMI 27–30:    {((pending["calculated_bmi"] > 27) & (pending["calculated_bmi"] <= 30)).sum():,}')
print(f'  BMI ≤ 27:     {(pending["calculated_bmi"] <= 27).sum():,}')
print(f'  BMI unknown:  {pending["calculated_bmi"].isna().sum():,}')

Both tasks done, no WM Rx: 366
  BMI > 30:     340
  BMI 27–30:    26
  BMI ≤ 27:     0
  BMI unknown:  0


In [15]:
# ── Enrollment trend ─────────────────────────────────────────────────────────
df['enroll_month'] = pd.to_datetime(
    df['subscription_start_date'], errors='coerce'
).dt.to_period('M').astype(str)
trend = df.groupby('enroll_month').size().reset_index(name='members_enrolled')
display(trend)

,enroll_month,members_enrolled
0,2025-10,1
1,2026-01,1458
2,2026-02,6732
3,2026-03,1125


In [16]:
# ── Drug switching preview ────────────────────────────────────────────────────
if switching_result:
    pivot, switch_summary, _ = switching_result
    display(switch_summary)
    display(pivot)

,Metric,Count,Pct
0,Total members with WM Rx,3009,
1,Stayed on same med,0,0.0%
2,Switched to different med,3009,100.0%


prescribed_drug_clean,Reported Drug (came in on),Bupropion Hcl Er (Sr) Oral Tablet Extended Release 12 Hour,Bupropion Hcl Er (Xl) Oral Tablet Extended Release 24 Hour,Liraglutide -Weight Management Subcutaneous Solution Pen-Injector,Naltrexone Hcl Oral Tablet,Topiramate Oral Tablet,Wegovy Oral Tablet,Wegovy Subcutaneous Solution Auto-Injector,Zepbound Subcutaneous Solution,Zepbound Subcutaneous Solution Auto-Injector,TOTAL
0,Mounjaro (Tirzepatide),0,0,0,0,0,0,0,0,20,20
1,My Medication Is Not Listed Here,0,0,0,0,0,1,3,0,3,7
2,Not Reported,31,2,1,5,28,3,26,1,185,282
3,Ozempic (Semaglutide),0,0,0,0,0,0,7,0,9,16
4,Victoza (Liraglutide),0,0,1,0,0,0,0,0,0,1
5,Wegovy (Semaglutide),1,0,0,0,1,6,627,0,278,913
6,Zepbound (Tirzepatida),0,0,0,0,0,0,0,0,1,1
7,Zepbound (Tirzepatide),0,0,0,0,0,2,11,0,1756,1769
8,TOTAL,32,2,2,5,29,12,674,1,2252,3009
